<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/27_transformer_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download data
!wget http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
!unzip -q spa-eng.zip

--2025-06-01 11:26:09--  http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.188.207, 192.178.163.207, 74.125.142.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.188.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2638744 (2.5M) [application/zip]
Saving to: ‘spa-eng.zip’

spa-eng.zip         100%[===================>]   2.52M  --.-KB/s    in 0.009s  

2025-06-01 11:26:09 (282 MB/s) - ‘spa-eng.zip’ saved [2638744/2638744]



In [2]:
import random

# Extract Data
text_file = "spa-eng/spa.txt"
with open(text_file) as f:
  lines = f.read().split("\n")[:-1]
text_pairs = []

for line in lines:
  english, spanish = line.split("\t")
  spanish = "[start] " + spanish + " [end]"
  text_pairs.append((english, spanish))

random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

In [3]:
# Vectorize the data
import tensorflow as tf
from tensorflow.keras import layers
import string
import re

# Handle special char not covered by strings.punctuation
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
  lowercase = tf.strings.lower(input_string)
  return tf.strings.regex_replace(
      lowercase, f"[{re.escape(strip_chars)}]", "")


vocab_size = 15000
sequence_length = 20

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    )

target_vectorization = layers.TextVectorization(
  max_tokens=vocab_size,
  output_mode="int",
  # Off set by one by one step during training
  output_sequence_length=sequence_length + 1,
  standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [4]:
# Create dataset
batch_size = 64

def format_dataset(eng, spa):
  eng = source_vectorization(eng)
  spa = target_vectorization(spa)
  # Why do we keep the original spanish sentence?
  return({
      "english":eng,
      "spanish":spa[:,:-1] # remove last token, to keep same len
  }), spa[:, 1:] # one token ahead

def make_dataset(pairs):
  eng_texts, spa_texts = zip(*pairs)
  eng_texts = list(eng_texts)
  spa_texts = list(spa_texts)
  dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
  dataset = dataset.batch(batch_size)
  dataset = dataset.map(format_dataset, num_parallel_calls=4)
  return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Transfomer Encoder

class TransformerEncoder(layers.Layer):
  def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
    super().__init__(**kwargs)
    self.embed_dim = embed_dim
    self.dense_dim = dense_dim
    self.num_heads = num_heads
    self.attention = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim
        )
    self.dense_proj = keras.Sequential(
        [layers.Dense(dense_dim, activation="relu"),layers.Dense(embed_dim),]
    )
    self.layernorm_1 = layers.LayerNormalization()
    self.layernorm_2 = layers.LayerNormalization()

  def call(self, inputs, mask=None):
    if mask is not None:
      mask = mask[:, tf.newaxis, :]
    attention_output = self.attention(
        inputs, inputs, attention_mask=mask)
    proj_input = self.layernorm_1(inputs + attention_output)
    proj_output = self.dense_proj(proj_input)
    return self.layernorm_2(proj_input + proj_output)

  def get_config(self):
    config = super().get_config()
    config.update({
      "embed_dim": self.embed_dim,
      "num_heads": self.num_heads,
      "dense_dim": self.dense_dim,
    })
    return config

# Positional Embedding
class PositionalEmbedding(layers.Layer):
  def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
    super().__init__(**kwargs)
    self.supports_masking = True
    self.token_embeddings = layers.Embedding(input_dim=input_dim,
                                             output_dim=output_dim,
                                             mask_zero=True)
    self.position_embeddings = layers.Embedding(input_dim=sequence_length,
                                                output_dim=output_dim)
    self.sequence_length = sequence_length
    self.input_dim = input_dim
    self.output_dim = output_dim

  def compute_mask(self, inputs, mask=None):
    return self.token_embeddings.compute_mask(inputs)

  def call(self, inputs):
    length = tf.shape(inputs)[-1]
    positions = tf.range(start=0, limit=length, delta=1)
    embedded_tokens = self.token_embeddings(inputs)
    embedded_positions = self.position_embeddings(positions)
    return embedded_tokens + embedded_positions

  def get_config(self):
    config = super().get_config()
    config.update({
        "output_dim": self.output_dim,
        "sequence_length": self.sequence_length,
        "input_dim": self.input_dim,})
    return config

In [9]:
# Decoder
class TransformerDecoder(layers.Layer):
  def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
    super().__init__(**kwargs)
    self.supports_masking = True
    self.embed_dim = embed_dim
    self.dense_dim = dense_dim
    self.num_heads = num_heads
    self.attention_1 = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim
        )

    self.attention_2 = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embed_dim
        )
    self.dense_proj = keras.Sequential(
        [layers.Dense(dense_dim, activation="relu"),layers.Dense(embed_dim),]
    )
    self.layernorm_1 = layers.LayerNormalization()
    self.layernorm_2 = layers.LayerNormalization()
    self.layernorm_3 = layers.LayerNormalization()


  def get_config(self):
    config = super().get_config()
    config.update({
      "embed_dim": self.embed_dim,
      "num_heads": self.num_heads,
      "dense_dim": self.dense_dim,
    })
    return config

  def get_causal_attention_mask(self, inputs):
    """
    Problem:
    Reminder: RNNs look at the input one step at the time.
    0 ... N to generate output step N(which is N+1 in the target sequence)

    Tranfoermer is order-agnostic: It can see the entire target sequence.
    Meaning, it for step N it would learn to copy N+1 in the output.
    In training that is no problem, but running inference the model is useless,
    because N+1 step is not given in inference.

    Fix:
    Mask the upper half of the pairwise attention matrix to prevent the model
    from paying attention to information from the future:
    0 ... N information will be available to generate N+1

    Example:
    Input : The cat sat
    Target: cat sat on
    step N=3, model should predict on

    RNN:
    Step 1: The predict cat
    Step 2: The cat predict sat
    Step 3: The cat sat predict on

    Transformer with no masking:
    Transformer see the entire target
    It learns to predict on by looking directly at on.
    In inference we only have The cat sat.
    The model dosent know how to predict, it was trained to look ahead.

    Masking fix:
    Step 1: atten to "The", rest masked
    Step 2: attend to The, cat, rest masked
    Step 3: atten to The, cat, sat, rest masked


    Mechanic:
    The Transformer attends to non masked tokens.
    Then outputs a prediction via softmax over the vocabulary.
    The prediction is compared to the true next word using loss
    Backpropagation is used to adjust the models weights.
    """
    input_shape = tf.shape(inputs)
    batch_size, sequence_length = input_shape[0], input_shape[1]
    i = tf.range(sequence_length)[:, tf.newaxis]
    j = tf.range(sequence_length)
    mask = tf.cast(i >= j, dtype="int32")
    mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1),
         tf.constant([1, 1], dtype=tf.int32)], axis=0)
    return tf.tile(mask, mult)

  def call(self, inputs, encoder_outputs, mask=None):
    casual_mask = self.get_causal_attention_mask(inputs)
    if mask is not None:
      padding_mask = tf.cast(
          mask[:, tf.newaxis, :], dtype="int32")
      padding_mask = tf.minimum(padding_mask, casual_mask)
    else:
      padding_mask = mask
    attention_output_1 = self.attention_1(
        query=inputs,
        value=inputs,
        key=inputs,
        attention_mask=casual_mask
    )
    attention_output_1 = self.layernorm_1(inputs + attention_output_1)
    attention_output_2 = self.attention_2(
        query=attention_output_1,
        value=encoder_outputs,
        key=encoder_outputs,
        attention_mask=padding_mask)
    attention_output_2 = self.layernorm_2(
        attention_output_1 + attention_output_2)
    proj_output = self.dense_proj(attention_output_2)
    return self.layernorm_3(attention_output_2 + proj_output)

In [10]:
embed_dim = 256
dense_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="spanish")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_outputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:938: UserWarning: Layer 'transformer_encoder' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [11]:
transformer.compile(
  optimizer="rmsprop",
  loss="sparse_categorical_crossentropy",
  metrics=["accuracy"])

transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 110s 70ms/step - accuracy: 0.1393 - loss: 4.7875 - val_accuracy: 0.2284 - val_loss: 2.8148
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 77s 59ms/step - accuracy: 0.2331 - loss: 2.7630 - val_accuracy: 0.2588 - val_loss: 2.2377
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2602 - loss: 2.2734 - val_accuracy: 0.2715 - val_loss: 2.0542
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2726 - loss: 2.0569 - val_accuracy: 0.2760 - val_loss: 1.9785
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2803 - loss: 1.9306 - val_accuracy: 0.2743 - val_loss: 2.0094
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2859 - loss: 1.8495 - val_accuracy: 0.2785 - val_loss: 1.9924
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2900 - loss: 1.7975 - val_accuracy: 0.2766 - val_loss: 2.0044
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 76s 59ms/step - accuracy: 0.2934 

In [12]:
import numpy as np
spa_vocab = target_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
  tokenized_input_sentence = source_vectorization([input_sentence])
  decoded_sentence = "[start]"
  for i in range(max_decoded_sentence_length):
    tokenized_target_sentence = target_vectorization(
        [decoded_sentence])[:, :-1]
    predictions = transformer(
        [tokenized_input_sentence, tokenized_target_sentence])
    sampled_token_index = np.argmax(predictions[0, i, :])
    sampled_token = spa_index_lookup[sampled_token_index]
    decoded_sentence += " " + sampled_token
    if sampled_token == "[end]":
      break
  return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
  input_sentence = random.choice(test_eng_texts)
  print("-")
  print(input_sentence)
  print(decode_sequence(input_sentence))

-
Look at this large map of America.
[start] mira este gran mapa de américa [end]
-
Any bed is better than no bed.
[start] cualquier cama es bueno que no [end]
-
Tom could swim a lot faster when he was young.
[start] tom podría nadar mucho más rápido cuando era joven [end]
-
You are naughty.
[start] son [UNK] [end]
-
I invited her to a movie.
[start] le pedí una película [end]
-
Tom was sentenced to five days in jail and a year on probation for drunken driving.
[start] a tom lo [UNK] a cinco días de prisión y un año de conducir completamente [UNK] [end]
-
Tom turned off the faucet.
[start] tom encendió la llave [end]
-
He accomplished it at last.
[start] Él la última vez en la última vez [end]
-
The wall is freshly painted.
[start] la pared es [UNK] de [UNK] [end]
-
It isn't much of a car.
[start] no es que se encontró un coche [end]
-
I hired an assistant.
[start] [UNK] a una asistente [end]
-
I often lie on this bench.
[start] a menudo lo [UNK] en esta mentira [end]
-
You're powerles